# Pipeline multiagente — vertical Discovery con modelo open-weight (Fase 2)

Ejecuta el DAG de Discovery con `Qwen/Qwen2.5-7B-Instruct` en int4 sobre una GPU T4/L4 de Colab. Ningún proveedor por API key.

**Runtime → Change runtime type → GPU (T4).**

In [ ]:
!git clone https://github.com/javierrosado/tesis.git /content/tesis
%cd /content/tesis
!pip -q install -r requirements.txt
!pip -q install torch transformers accelerate bitsandbytes outlines peft

In [ ]:
# opcional: token de Hugging Face para modelos con acceso restringido o para subir resultados
import os
# os.environ['HF_TOKEN'] = '...'
os.environ['PIPELINE_PROVIDER'] = 'hf_local'

In [ ]:
# prueba de humo del proveedor: carga el modelo y pide un JSON trivial
import sys; sys.path.insert(0,'/content/tesis')
from core.registry import Registry
reg = Registry('/content/tesis')
ad = reg.adapter_for('product-analyst-agent')
print(ad.capabilities())
r = ad.complete([{'role':'user','content':'Devuelve {"ok": true}'}], schema={'type':'object','properties':{'ok':{'type':'boolean'}},'required':['ok']})
print(r.model_id, r.text, r.latency_ms, 'ms')

In [ ]:
# corrida del vertical para un requerimiento
!python scripts/run_pipeline.py --req SIM-P05-R001 --telemetry telemetry/hf_local_R001.jsonl

In [ ]:
# corrida completa (todos los requerimientos presentes en data/requerimientos)
# !python scripts/run_pipeline.py --all --telemetry telemetry/hf_local_all.jsonl

In [ ]:
# resumen de telemetría: rechazos de contrato, rondas de gobernanza, tokens, latencia
import json, collections
ev=[json.loads(l) for l in open('telemetry/hf_local_R001.jsonl')]
c=collections.Counter(e['event'] for e in ev); print(dict(c))
print('tokens in/out:', sum(e.get('tokens_in',0) for e in ev), sum(e.get('tokens_out',0) for e in ev))
print('modelos usados:', {e.get('model_id') for e in ev if 'model_id' in e})

## Siguiente paso (Fase 3)
`notebooks/finetune_qlora.ipynb`: genera el dataset sintético de `product-analyst-agent`, entrena un adaptador LoRA, lo publica en HF Hub y lo declara en `config/providers.yaml` → `overrides`. Luego se repite esta corrida y se compara la telemetría (ablación).